# CMSC 173 &middot; Module 12 &middot; Example 1
## One network, by hand

Two inputs, two hidden sigmoid neurons, one sigmoid output: 9 parameters. One forward pass, one backward pass, one gradient step, and a numerical check of the gradient.

**How to use this notebook.** Runtime &rarr; Run all. Every number printed below is on the
slides (scikit-learn 1.4.2; a newer version can differ slightly for the trained networks);
change something and see what moves.

**One neuron.** $z = 0.2(0.5) + 0.4(0.8) + 0.1 = 0.52$, then the sigmoid.

In [ ]:
import numpy as np

sigmoid = lambda z: 1 / (1 + np.exp(-z))
print(round(0.2 * 0.5 + 0.4 * 0.8 + 0.1, 2), round(sigmoid(0.52), 3))      # 0.52 0.627

**The forward pass.** Row $j$ of `W1` holds the weights into hidden neuron $j$.

In [ ]:
x = np.array([0.5, 0.8]); y = 1.0
W1 = np.array([[0.2, 0.4], [0.3, 0.1]]); b1 = np.array([0.1, 0.2])
W2 = np.array([0.6, 0.5]); b2 = 0.3
z1 = W1 @ x + b1; a1 = sigmoid(z1)            # hidden: z = 0.52, 0.43 -> a = 0.627, 0.606
z2 = W2 @ a1 + b2; y_hat = sigmoid(z2)       # output: z = 0.979 -> 0.727
L = 0.5 * (y_hat - y) ** 2
print(z1.round(2), a1.round(3), round(z2, 3), round(y_hat, 4), round(L, 4))

**Backpropagation.** The error at the output, the errors at the hidden neurons, then every gradient is an error times the input at the start of its connection.

In [ ]:
d2 = (y_hat - y) * y_hat * (1 - y_hat)       # output error: -0.0542
d1 = W2 * d2 * a1 * (1 - a1)                 # hidden errors: -0.0076, -0.0065
gW2, gb2 = d2 * a1, d2                       # -0.034, -0.0328 | -0.0542
gW1, gb1 = np.outer(d1, x), d1               # [[-0.0038, -0.0061], [-0.0032, -0.0052]]
print(round(d2, 4), d1.round(4))
print(gW2.round(4), round(gb2, 4), gW1.round(4).tolist(), gb1.round(4))

**One gradient step**, learning rate 0.5, on all 9 parameters.

In [ ]:
alpha = 0.5
W1n, b1n, W2n, b2n = W1 - alpha * gW1, b1 - alpha * gb1, W2 - alpha * gW2, b2 - alpha * gb2
y_new = sigmoid(W2n @ sigmoid(W1n @ x + b1n) + b2n)
print(W2n.round(4), round(b2n, 4), round(W1n[0, 1], 4))                  # [0.617 0.5164] 0.3271 0.403
print(round(y_new, 4), round(0.5 * (y_new - 1) ** 2, 4))                  # 0.7367 0.0347: loss down 7%

**Gradient checking.** Nudge one weight up and down and measure the loss.

In [ ]:
def loss_with(w):
    return 0.5 * (sigmoid(w * a1[0] + W2[1] * a1[1] + b2) - y) ** 2

h = 1e-4
print(round(loss_with(0.6 + h), 8), round(loss_with(0.6 - h), 8))       # 0.03727352 0.03728032
print(round((loss_with(0.6 + h) - loss_with(0.6 - h)) / (2 * h), 4))     # -0.034, as backpropagation says

**Try it.** Set the target to `y = 0.0` and rerun. Which way does every gradient point now? (Review question: the output error becomes +0.144.)